# MLB Project 3 - RAG Chatbot

## Project Overview

Welcome to Project 3! In this project, you'll build a **Retrieval-Augmented Generation (RAG)** chatbot that can answer questions about a PDF document.

### What is RAG?
RAG combines two powerful concepts:
1. **Retrieval**: Finding relevant information from a document
2. **Generation**: Using an LLM to generate natural language answers

### What You'll Learn
- How to extract and process text from PDFs
- How to create text embeddings (vector representations)
- How to build a simple vector database
- How to search for relevant information using similarity
- How to use an LLM to generate answers based on context

### Project Structure
1. Setup and imports
2. Build a Vector Database
3. PDF processing utilities
4. Question answering system
5. Put it all together!

---

## Step 1: Setup and Imports

First, let's install the required libraries and import them.

In [ ]:
!pip install sentence-transformers pypdf transformers huggingface_hub torch tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.3/331.3 kB 8.6 MB/s eta 0:00:00


In [ ]:
# Import all necessary libraries
import os
import numpy as np
import warnings
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from transformers import pipeline
from huggingface_hub import login

# Suppress unnecessary warnings for cleaner output
warnings.filterwarnings("ignore", category=UserWarning)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Step 2: Configuration

Let's set up our model names and API keys.

In [ ]:
# Configuration settings
LLM_MODEL = "google/flan-t5-base"  # The language model for generating answers
EMBEDDING_MODEL = "all-MiniLM-L12-v2"  # The model for creating embeddings
HF_API_KEY = os.getenv("HF_API_KEY", "YOUR-HF-API-KEY-HERE")  # Optional Hugging Face API key

print(f"📋 LLM Model: {LLM_MODEL}")
print(f"📋 Embedding Model: {EMBEDDING_MODEL}")

📋 LLM Model: google/flan-t5-base
📋 Embedding Model: all-MiniLM-L12-v2


## Step 3: Build the Vector Database Class

A **Vector Database** stores text as numerical vectors (embeddings) and allows us to search for similar text using mathematical operations.

### What are Embeddings?
Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings.

Example:
- "dog" and "puppy" would have similar embeddings
- "dog" and "car" would have very different embeddings

### 3.1: Initialize the Vector Database

**TODO**: Complete the `__init__` method to:
1. Load the SentenceTransformer model
2. Get the embedding dimension
3. Initialize empty storage for embeddings and text

In [ ]:
class VectorDB:
    def __init__(self, model_name: str):
        """
        Initialize the Vector Database with an embedding model.

        Args:
            model_name: Name of the SentenceTransformer model to use
        """
        self.embedModel = SentenceTransformer(model_name)

        self.embed_size = self.embedModel.get_sentence_embedding_dimension()

        self._embeddings = np.empty((0, self.embed_size))

        self._strings = []

        print(f"✅ VectorDB initialized with embedding dimension: {self.embed_size}")

### 3.2: Add Data to the Database

**TODO**: Complete the `addToDatabase` method to:
1. Convert text strings to embeddings
2. Store the embeddings in the database
3. Store the original text strings

In [ ]:
    def addToDatabase(self, input: list[str]):
        """
        Add text chunks to the vector database.

        Args:
            input: List of text strings to add to the database
        """
        new_embeddings = self.embedModel.encode(input)

        if self._embeddings.shape[0] == 0:
            self._embeddings = new_embeddings
        else:
            self._embeddings = np.vstack((self._embeddings, new_embeddings))

        self._strings.extend(input)

        print(f"✅ Added {len(input)} chunks. Total chunks: {len(self._strings)}")

# Add this method to the VectorDB class
VectorDB.addToDatabase = addToDatabase

### 3.3: Clear the Database

**TODO**: Complete the `clearDatabase` method to reset the database.

In [ ]:
    def clearDatabase(self):
        """
        Clear all data from the vector database.
        """
        self._embeddings = np.empty((0, self.embed_size))
        self._strings = []

        print("🗑️ Database cleared!")

# Add this method to the VectorDB class
VectorDB.clearDatabase = clearDatabase

### 3.4: Calculate Euclidean Similarity

**TODO**: Implement the similarity function.

**What is Euclidean Distance?**
It's the straight-line distance between two points in space. We convert it to similarity:
- Distance = 0 → Similarity = 1 (identical)
- Distance = large → Similarity = close to 0 (very different)

In [ ]:
    def euclideanSim(self, x, y):
        """
        Calculate Euclidean similarity between two vectors.

        Args:
            x: First vector (numpy array)
            y: Second vector (numpy array)

        Returns:
            Similarity score (higher = more similar)
        """
        distance = np.linalg.norm(x - y)

        similarity = 1 / (1 + distance)

        return similarity

# Add this method to the VectorDB class
VectorDB.euclideanSim = euclideanSim

### 3.5: Search the Database

**TODO**: Implement the search functionality to find the most similar text chunks.

In [ ]:
    def search(self, query: str, n_return=3):
        """
        Search the database for the most similar chunks to the query.

        Args:
            query: The search query string
            n_return: Number of top results to return

        Returns:
            Tuple of (text_chunks, similarity_scores)
        """

        query_embedding = self.embedModel.encode([query])[0]

        similarities = [self.euclideanSim(query_embedding, emb) for emb in self._embeddings]

        top_indices = np.argsort(similarities)[::-1][:n_return]

        top_chunks = [self._strings[i] for i in top_indices]
        top_scores = [similarities[i] for i in top_indices]

        return top_chunks, top_scores

# Add this method to the VectorDB class
VectorDB.search = search

### Test the Vector Database

Let's test our VectorDB with some sample data!

In [ ]:
# Test the VectorDB
print("Testing VectorDB...\n")

# Create a test database
test_vdb = VectorDB(EMBEDDING_MODEL)

# Add some test data
test_data = [
    "Python is a programming language.",
    "Machine learning involves training models on data.",
    "Dogs are loyal pets.",
    "Neural networks are inspired by the human brain."
]

test_vdb.addToDatabase(test_data)

# Search for something
query = "What is ML?"
results, scores = test_vdb.search(query, n_return=2)

print(f"\n🔍 Query: '{query}'\n")
for i, (chunk, score) in enumerate(zip(results, scores), 1):
    print(f"Result {i} (similarity: {score:.4f}):")
    print(f"  {chunk}\n")

Testing VectorDB...



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ VectorDB initialized with embedding dimension: 384
✅ Added 4 chunks. Total chunks: 4

🔍 Query: 'What is ML?'

Result 1 (similarity: 0.4539):
  Machine learning involves training models on data.

Result 2 (similarity: 0.4379):
  Neural networks are inspired by the human brain.



## Step 4: PDF Processing Utilities

Now we'll build functions to extract and process text from PDF files.

### 4.1: Clean Text Function

This function removes extra whitespace and newlines from text.

In [ ]:
def clean_text(text: str) -> str:
    """
    Clean text by removing extra whitespace and newlines.

    Args:
        text: Raw text string

    Returns:
        Cleaned text string
    """
    # Split text into words and join with single spaces
    return " ".join(text.split())

# Test the function
test_text = "This   has    extra\n\nspaces   and\nnewlines."
print(f"Original: {repr(test_text)}")
print(f"Cleaned:  {repr(clean_text(test_text))}")

Original: 'This   has    extra\n\nspaces   and\nnewlines.'
Cleaned:  'This has extra spaces and newlines.'


### 4.2: Create Text Chunks

**TODO**: Split long text into overlapping chunks.

**Why Overlapping Chunks?**
- Ensures we don't split important information across boundaries
- Example: chunk_size=500, overlap=50 means each chunk shares 50 characters with the next

In [ ]:
def chunksFromText(text: str, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks.

    Args:
        text: Input text string
        chunk_size: Size of each chunk in characters
        overlap: Number of overlapping characters between chunks

    Returns:
        List of text chunks
    """
    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(text), step):
        chunks.append(text[i:i+chunk_size])

    return chunks

# Test the function
test_text = "A" * 100  # 100 characters
test_chunks = chunksFromText(test_text, chunk_size=30, overlap=5)
print(f"Text length: {len(test_text)}")
print(f"Number of chunks: {len(test_chunks)}")
print(f"First chunk length: {len(test_chunks[0]) if test_chunks else 0}")

Text length: 100
Number of chunks: 4
First chunk length: 30


### 4.3: Process PDF and Add to Database

**TODO**: Read a PDF, extract text, create chunks, and add to the database.

In [ ]:
def chunksFromPDF(vDB, path: str, startPage=0, endPage=None):
    """
    Extract text from a PDF, chunk it, and add to the vector database.

    Args:
        vDB: VectorDB instance to add chunks to
        path: Path to the PDF file
        startPage: First page to process (0-indexed)
        endPage: Last page to process (None = all pages)
    """
    reader = PdfReader(path)

    pages = reader.pages[startPage:endPage] if endPage else reader.pages[startPage:]

    print(f"📄 Processing {len(pages)} pages from PDF...")

    all_chunks = []

    for page_num, page in enumerate(tqdm(pages, desc="Extracting text"), startPage):
        text = page.extract_text()

        text = clean_text(text)

        if len(text) < 100:
            continue

        page_chunks = chunksFromText(text)

        page_chunks = [f"[Page {page_num+1}] {chunk}" for chunk in page_chunks]
        all_chunks.extend(page_chunks)

    vDB.addToDatabase(all_chunks)

    print(f"✅ Successfully processed PDF: {len(all_chunks)} chunks added to database")

## Step 5: Question Answering System

Now we'll create the function that ties everything together!

### 5.1: Generate Answer Function

**TODO**: Implement the RAG pipeline:
1. Retrieve relevant context from the database
2. Create a prompt with context and question
3. Generate an answer using the LLM

In [ ]:
def generateAnswer(question: str, vDB, llm):
    """
    Generate an answer to a question using RAG.

    Args:
        question: User's question
        vDB: VectorDB instance with loaded documents
        llm: Language model pipeline for generation

    Returns:
        Generated answer string
    """
    relevant_chunks, scores = vDB.search(question, n_return=3)

    context = "\n\n".join(relevant_chunks)

    prompt = f"""
Based on the following context, answer the question.

Context:
{context}

Question: {question}

Answer:
"""

    result = llm(prompt, max_new_tokens=150, max_length=None, num_return_sequences=1)

    generated_text = result[0]['generated_text']

    answer = generated_text.split("Answer:")[-1].strip()

    return answer

## Step 6: Put It All Together! 🎉

Now let's run the complete RAG chatbot!

### 6.1: Login to Hugging Face (Optional)

If you have an API key, this step helps avoid rate limits.

In [ ]:
print("Logging in to Hugging Face Hub...")
try:
    login(token=HF_API_KEY)
    print("✅ Login successful!")
except Exception as e:
    print(f"⚠️ Skipping login (API key optional): {e}")

### 6.2: Initialize the Vector Database

**TODO**: Create your VectorDB instance.

In [ ]:
print("Loading embedding model...")

# TODO: Create an instance of VectorDB using EMBEDDING_MODEL
# Hint: vDB = VectorDB(EMBEDDING_MODEL)
# TODO: Create an instance of VectorDB using EMBEDDING_MODEL
vDB = VectorDB(EMBEDDING_MODEL)

print("✅ Vector Database ready!")

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ VectorDB initialized with embedding dimension: 384
✅ Vector Database ready!


### 6.3: Load and Process the PDF

**TODO**: Process the TechNova IT Handbook PDF.

**Note**: Make sure the PDF file is in the same directory as this notebook!

In [ ]:
# Locate the PDF file
pdf_path = "TechNova_IT_Handbook.pdf"

# Check if file exists
if not os.path.exists(pdf_path):
    print(f"❌ PDF not found at {pdf_path}")
    print("Please make sure 'TechNova_IT_Handbook.pdf' is in the same folder as this notebook.")
else:
    print(f"📄 Found PDF: {pdf_path}")

    chunksFromPDF(vDB, pdf_path)

📄 Found PDF: TechNova_IT_Handbook.pdf
📄 Processing 21 pages from PDF...


Extracting text: 100%|██████████| 21/21 [00:00<00:00, 304.90it/s]

✅ Added 40 chunks. Total chunks: 40
✅ Successfully processed PDF: 40 chunks added to database


### 6.4: Load the Language Model

**TODO**: Load the LLM for generating answers.

**Note**: This may take a few minutes the first time!

In [ ]:
print("Loading LLM model...")
print("⏳ This may take a few minutes...")

llm = pipeline("text-generation", model=LLM_MODEL, device=0)

print("✅ LLM loaded successfully!")

Loading LLM model...
⏳ This may take a few minutes...


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

✅ LLM loaded successfully!


### 6.5: Interactive Q&A Session

**TODO**: Create an interactive loop for asking questions.

Try asking questions like:
- "What is the password policy?"
- "How do I report a security incident?"
- "What are the remote work guidelines?"

In [ ]:
print("\n" + "="*50)
print("🤖 RAGBot is ready!")
print("Ask questions about the TechNova IT Handbook.")
print("Type 'exit' or 'quit' to stop.")
print("="*50 + "\n")

while True:
    # Get user input
    question = input("\n❓ Your question: ")

    # Check if user wants to exit
    if question.lower() in ["exit", "quit"]:
        print("\n👋 Goodbye! Thanks for using RAGBot!")
        break

    # Skip empty questions
    if not question.strip():
        continue

    print("\n🤔 Thinking...\n")
    answer = generateAnswer(question, vDB, llm)

    print(f"💡 Answer: {answer}")
    print("\n" + "-"*50)


🤖 RAGBot is ready!
Ask questions about the TechNova IT Handbook.
Type 'exit' or 'quit' to stop.


❓ Your question: What is the password policy?


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



🤔 Thinking...



Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💡 Answer: 

--------------------------------------------------

❓ Your question: exit

👋 Goodbye! Thanks for using RAGBot!


### Alternative: Single Question Testing

If you prefer to test with individual questions, use this cell instead:

In [ ]:
# Test with a single question
test_question = "What is the password policy?"

print(f"Question: {test_question}\n")
answer = generateAnswer(test_question, vDB, llm)
print(f"Answer: {answer}")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the password policy?

Answer: 


## 🎓 Congratulations!

You've successfully built a RAG chatbot! Here's what you accomplished:

1. ✅ Created a vector database to store document embeddings
2. ✅ Implemented similarity search using Euclidean distance
3. ✅ Processed PDF documents and created text chunks
4. ✅ Built a complete RAG pipeline for question answering
5. ✅ Integrated an LLM to generate natural language responses

### 🚀 Next Steps

Want to improve your RAG bot? Try:
- Experimenting with different chunk sizes and overlap values
- Using cosine similarity instead of Euclidean distance
- Adding more sophisticated text preprocessing
- Trying different embedding models
- Implementing a better prompt engineering strategy
- Adding source citations to your answers

### 📚 Additional Resources

- [SentenceTransformers Documentation](https://www.sbert.net/)
- [Hugging Face Transformers](https://huggingface.co/docs/transformers/)
- [RAG Paper (Lewis et al.)](https://arxiv.org/abs/2005.11401)

Great work! 🎉